# Structural Optimization

In [ ]:
import pandas as pd
import numpy as np
import math
from anastruct import SystemElements
import matplotlib.pyplot as plt

# Load filtered data from CSV
db_filtered = pd.read_csv('db_filtered.csv')

def check_shape(Ag, r_min, Pu_kips, L_in, Fy=50, E=29000):
    klr = L_in / r_min
    if Pu_kips < 0 and klr > 200: return False, 0.0, klr
    if Pu_kips >= 0 and klr > 300: return False, 0.0, klr
    phi_t = 0.9
    if phi_t * Fy * Ag < abs(Pu_kips): return False, 0.0, klr
    if Pu_kips < 0: 
        Fe = (math.pi**2 * E) / (klr**2)
        sl_limit = 4.71 * math.sqrt(E / Fy)
        Fcr = (0.658**(Fy / Fe)) * Fy if klr <= sl_limit else 0.877 * Fe
        phi_c = 0.9
        if phi_c * Fcr * Ag < abs(Pu_kips):
            return False, phi_c * Fcr * Ag, klr
    return True, 0.0, klr

def select_candidates(Pu_kN, L_m, family='HSS'):
    if db_filtered is None: return []
    Pu_kips = Pu_kN * 0.224809
    L_in = L_m * 39.3701
    subset = db_filtered[db_filtered['Type'] == family] if family else db_filtered
    valid = []
    for _, row in subset.iterrows():
        fy = 46 if row['Type'].startswith('HSS') else (36 if row['Type'] == 'L' else 50)
        passes, cap, klr = check_shape(row['A'], row['r_min'], Pu_kips, L_in, Fy=fy)
        if passes:
            valid.append({'Label': str(row['AISC_Manual_Label']), 'Weight': float(row['W']), 'Area': float(row['A']), 'Ix': float(row['Ix']), 'KL/r': float(klr)})
    return sorted(valid, key=lambda x: x['Weight'])

def select_lightest(Pu_kN, L_m, family='HSS'):
    candidates = select_candidates(Pu_kN, L_m, family)
    return candidates[0] if candidates else None

def build_truss(results_map):
    ss = SystemElements()
    N, depth, dx, dy = 16, 0.6, 1.16875, 1.0
    dy_per_panel = dy / N
    chord_bot, chord_top, web_vert, web_diag = [], [], [], []
    for i in range(N):
        chord_bot.append(ss.add_truss_element(location=[[i*dx, 4 + i*dy_per_panel], [(i+1)*dx, 4 + (i+1)*dy_per_panel]]))
        chord_top.append(ss.add_truss_element(location=[[i*dx, 4 + depth + i*dy_per_panel], [(i+1)*dx, 4 + depth + (i+1)*dy_per_panel]]))
    for i in range(N + 1):
        web_vert.append(ss.add_truss_element(location=[[i*dx, 4 + i*dy_per_panel], [i*dx, 4 + depth + i*dy_per_panel]]))
    for i in range(N):
        web_diag.append(ss.add_truss_element(location=[[i*dx, 4 + i*dy_per_panel], [(i+1)*dx, 4 + depth + (i+1)*dy_per_panel]]))
    for sx in [0.0, 18.7]:
        ss.add_support_hinged(node_id=ss.find_node_id([sx, 4 + (sx/dx)*(dy_per_panel)]))
    mapping = {"Top Chord": chord_top, "Bottom Chord": chord_bot, "Vertical Webs": web_vert, "Diagonal Webs": web_diag}
    for g, r in results_map.items():
        if not r: continue
        a, i, e = float(r['Area']*0.00064516), float(r['Ix']*4.1623e-7), 200e9
        for eid in mapping[g]:
            ss.element_map[eid].EA, ss.element_map[eid].EI = a*e, i*e
    for eid in chord_top: ss.q_load(q=-2.3, element_id=eid, direction='y')
    ss.q_load(q=3.2, element_id=chord_bot[0], direction='x')
    return ss

def get_max_group_forces(system, ids):
    forces = []
    for eid in ids:
        el = system.element_map[eid]
        if hasattr(el, 'axial_force') and el.axial_force is not None:
            forces.extend([el.axial_force[0], el.axial_force[1]])
        else:
            forces.append(0)
    return max(forces, key=abs) if forces else 0

def apply_member_selection(system, groups):
    print("\n--- MEMBER SELECTION ---")
    results = {}
    for name, data in groups.items():
        max_p = get_max_group_forces(system, data['ids'])
        shape = select_lightest(max_p, data['L'], family=data['family'])
        if shape:
            results[name] = shape
            print(f"{name:20} | {shape['Label']:15} | {shape['Weight']:5.1f} plf | {max_p:7.2f} kN")
            a_si, i_si, e_si = float(shape['Area'] * 0.00064516), float(shape['Ix'] * 4.1623e-7), 200e9
            for eid in data['ids']:
                system.element_map[eid].EA, system.element_map[eid].EI = a_si * e_si, i_si * e_si
        else:
            print(f"{name:20} | ERROR")
    return results


## 1. Longitudinal Truss

In [ ]:

truss_results = {'Top Chord': {'Label': 'WT3X8', 'Weight': 8.0, 'Area': 2.37, 'Ix': 1.69, 'KL/r': 54.57703436018957}, 'Bottom Chord': {'Label': 'WT3X8', 'Weight': 8.0, 'Area': 2.37, 'Ix': 1.69, 'KL/r': 54.57703436018957}, 'Vertical Webs': {'Label': 'L2X2X1/8', 'Weight': 1.65, 'Area': 0.491, 'Ix': 0.189, 'KL/r': 60.41447570332481}, 'Diagonal Webs': {'Label': 'L2-1/2X2X3/16', 'Weight': 2.75, 'Area': 0.818, 'Ix': 0.511, 'KL/r': 121.06767840375588}}
ss = build_truss(truss_results)
ss.solve()
disp = ss.system_displacement_vector
max_uy = 0
for nid in ss.node_map:
    uy = disp[(nid-1)*3 + 1]
    if abs(uy) > abs(max_uy): max_uy = uy
ratio = abs(18.7 / max_uy) if max_uy != 0 else 0
print(f"LONGITUDINAL TRUSS (OPTIMIZED)")
print(f"Max Deflection: {max_uy*1000:.2f} mm | Ratio: L/{ratio:.0f}")


In [ ]:
ss.show_displacement()

## 2. Transverse Stiffening Truss

In [ ]:

ts = SystemElements()
L_trans, N_trans, depth_trans = 20.675, 20, 0.6
dx_trans = L_trans / N_trans
for i in range(N_trans):
    ts.add_truss_element(location=[[i*dx_trans, 0], [(i+1)*dx_trans, 0]])
    ts.add_truss_element(location=[[i*dx_trans, depth_trans], [(i+1)*dx_trans, depth_trans]])
for i in range(N_trans + 1):
    ts.add_truss_element(location=[[i*dx_trans, 0], [i*dx_trans, depth_trans]])
for i in range(N_trans):
    ts.add_truss_element(location=[[i*dx_trans, 0], [(i+1)*dx_trans, depth_trans]])
ts.add_support_hinged(node_id=ts.find_node_id([0, 0]))
ts.add_support_roll(node_id=ts.find_node_id([L_trans, 0]))
# Transfer load 21.5 kN
for i in range(11):
    ts.point_load(Fy=-21.5, node_id=ts.find_node_id([i * 2.0675, 0]))
ts.solve()
ts_groups = {
    "TS Top": {"ids": list(range(2, 41, 2)), "L": dx_trans, "family": "HSS"},
    "TS Bot": {"ids": list(range(1, 41, 2)), "L": dx_trans, "family": "HSS"},
    "TS Vert": {"ids": list(range(41, 62)), "L": depth_trans, "family": "L"},
    "TS Diag": {"ids": list(range(62, 82)), "L": math.sqrt(dx_trans**2 + depth_trans**2), "family": "L"},
}
apply_member_selection(ts, ts_groups)
ts.solve()


In [ ]:
ts.show_displacement()